# NOVA Remote GPU Worker — Google Colab

Этот ноутбук превращает бесплатную Colab GPU-сессию во временный удалённый worker для NOVA Motion + VFX Studio.

**Схема:** iPhone → NOVA → HTTPS tunnel → Colab GPU → Blender / FFmpeg / WanGP handoff → готовый MP4 обратно в NOVA.

Платные API не используются. Colab может отключить runtime или не выдать GPU.

In [ ]:
# 1) Проверка GPU и базовые настройки
import os, secrets, subprocess, sys, time, re, json, shutil
from pathlib import Path

subprocess.run(['nvidia-smi'], check=False)
PORT = 7861
USE_DRIVE_MIRROR = False   # True = копировать completed jobs в MyDrive/NOVA_RENDER_QUEUE
TOKEN = secrets.token_urlsafe(24)
print('NOVA token created:', TOKEN)


In [ ]:
# 2) Установка бесплатного worker stack, Blender, FFmpeg и pose tracker
env=os.environ.copy(); env['DEBIAN_FRONTEND']='noninteractive'
subprocess.run(['sudo','apt-get','update','-qq'],check=True,env=env)
subprocess.run(['sudo','apt-get','install','-y','--no-install-recommends','blender','ffmpeg'],check=True,env=env)
subprocess.run([sys.executable,'-m','pip','install','-q','fastapi','uvicorn','python-multipart','mediapipe','opencv-python-headless','gradio_client'],check=True)

REPO=Path('/content/nova-robot')
if (REPO/'.git').exists():
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'],check=True)
    subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'],check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch','main','https://github.com/magomedt149/nova-robot.git',str(REPO)],check=True)
print('Repo ready:', REPO)
subprocess.run(['blender','--version'],check=False)
subprocess.run(['ffmpeg','-version'],check=False,stdout=subprocess.DEVNULL)


In [ ]:
# 3) (Опционально) Google Drive как резервное хранилище результатов
if USE_DRIVE_MIRROR:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mirror:', '/content/drive/MyDrive/NOVA_RENDER_QUEUE/completed')
else:
    print('Drive mirror выключен. Файлы возвращаются напрямую в NOVA через tunnel.')


In [ ]:
# 4) Запуск NOVA worker + бесплатный Cloudflare Quick Tunnel
import urllib.request
cloudflared=Path('/content/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)

worker_log=Path('/content/nova_worker.log')
tunnel_log=Path('/content/nova_tunnel.log')
os.environ['NOVA_REMOTE_TOKEN']=TOKEN
os.environ['NOVA_REMOTE_JOB_ROOT']='/content/NOVA_REMOTE_JOBS'
if USE_DRIVE_MIRROR:
    os.environ['NOVA_REMOTE_DRIVE_ROOT']='/content/drive/MyDrive/NOVA_RENDER_QUEUE'

worker = subprocess.Popen(
    [sys.executable, str(REPO/'automation/remote_gpu_worker.py'), '--host','0.0.0.0','--port',str(PORT)],
    stdout=worker_log.open('w'), stderr=subprocess.STDOUT, env=os.environ.copy())
time.sleep(3)
if worker.poll() is not None:
    print(worker_log.read_text(errors='replace'))
    raise RuntimeError('NOVA worker did not start')

tunnel = subprocess.Popen(
    [str(cloudflared),'tunnel','--url',f'http://127.0.0.1:{PORT}','--no-autoupdate'],
    stdout=tunnel_log.open('w'), stderr=subprocess.STDOUT, text=True)

url=None
for _ in range(60):
    time.sleep(1)
    text=tunnel_log.read_text(errors='replace') if tunnel_log.exists() else ''
    m=re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', text)
    if m:
        url=m.group(0); break
if not url:
    print(tunnel_log.read_text(errors='replace'))
    raise RuntimeError('Cloudflare tunnel URL was not found')

print('\n'+'='*76)
print('NOVA WORKER URL:', url)
print('NOVA TOKEN     :', TOKEN)
print('='*76)
print('На iPhone открой Motion + VFX → Remote GPU и вставь эти два значения.')


In [ ]:
# 5) Проверка worker из Colab
import requests
r=requests.get(url+'/health', headers={'X-NOVA-Token':TOKEN}, timeout=20)
print(json.dumps(r.json(), ensure_ascii=False, indent=2))


## WanGP

Blender и FFmpeg jobs выполняются полностью автоматически через NOVA. Для WanGP worker автоматически готовит `WAN_GP_INPUT.mp4`, prompt и `WANGP_HANDOFF.json`.

Полностью автоматический WanGP включается только если известен и настроен совместимый Gradio API endpoint. Иначе статус будет `waiting_wangp`: запускается проверенный WanGP UI, после генерации файл можно финализировать обратно в worker.

In [ ]:
# 6) (Опционально) Подготовить и запустить проверенный WanGP UI
USE_WANGP = False
if USE_WANGP:
    import json
    UPSTREAM_DIR=Path('/content/Wan2GP-on-Colab')
    UPSTREAM_COMMIT='e428b5ebc0d49589474ef5d81e05cc2ab3c1e17b'
    if not (UPSTREAM_DIR/'.git').exists():
        subprocess.run(['git','clone','https://github.com/Square-Zero-Labs/Wan2GP-on-Colab.git',str(UPSTREAM_DIR)],check=True)
    subprocess.run(['git','-C',str(UPSTREAM_DIR),'fetch','--depth','1','origin',UPSTREAM_COMMIT],check=True)
    subprocess.run(['git','-C',str(UPSTREAM_DIR),'checkout','--detach',UPSTREAM_COMMIT],check=True)
    upstream_notebook=UPSTREAM_DIR/'wan2gp-google-colab.ipynb'
    upstream=json.loads(upstream_notebook.read_text(encoding='utf-8'))
    launch_code=None
    for number, cell in enumerate(upstream['cells']):
        if cell.get('cell_type')!='code': continue
        code=''.join(cell.get('source') or [])
        if 'Launching Wan2GP' in code:
            launch_code=code; continue
        code=code.replace('USE_GOOGLE_DRIVE_DATA = False', f'USE_GOOGLE_DRIVE_DATA = {USE_DRIVE_MIRROR!r}')
        print(f'--- WanGP setup cell {number} ---')
        exec(compile(code, f'{upstream_notebook.name}:cell_{number}', 'exec'), globals())
    if launch_code is None: raise RuntimeError('WanGP launch cell not found')
    print('WanGP prepared. Run next cell to open its UI.')
else:
    print('WanGP setup skipped. Set USE_WANGP=True and rerun this cell when needed.')


In [ ]:
# 7) (Опционально) Открыть WanGP Gradio UI
if globals().get('USE_WANGP') and globals().get('launch_code'):
    exec(compile(launch_code, 'verified_wan2gp_launch_cell', 'exec'), globals())
else:
    print('WanGP не подготовлен — предыдущая клетка была пропущена.')


In [ ]:
# 8) Финализировать вручную созданный WanGP-файл обратно в NOVA (если нужно)
JOB_ID = ''
WAN_RESULT = ''  # например /content/Wan2GP/outputs/result.mp4
if JOB_ID and WAN_RESULT:
    p=Path(WAN_RESULT)
    if not p.is_file(): raise FileNotFoundError(p)
    with p.open('rb') as f:
        rr=requests.post(url+f'/jobs/{JOB_ID}/finalize', headers={'X-NOVA-Token':TOKEN}, files={'result':(p.name,f,'video/mp4')}, timeout=600)
    print(rr.status_code, rr.text)
else:
    print('Заполни JOB_ID и WAN_RESULT только после ручного WanGP render.')
